# Lekcija 01 - Uvod v AI agente

Dobrodošli na prvi lekciji v tečaju **AI agenti za začetnike**!

**AI agent** je program, ki uporablja velik jezikovni model (LLM) kot svoj mehanizem sklepanja in lahko v resničnem svetu izvaja *dejanja* — kliče API-je, poizveduje po podatkovnih bazah ali izvaja kodo — da doseže cilj v imenu uporabnika.

V tem zvezku boste zgradili svojega prvega agenta: **potovalni agent**, ki priporoča počitniške destinacije. Med tem boste izvedeli, kako:

1. Povezati se z Azure AI Foundry Agent Service z uporabo **Microsoft Agent Framework**.
2. Agentu dati **orodje** — preprosto funkcijo v Pythonu, ki jo lahko kliče.
3. Zagnati agenta in pregledati njegov odgovor.
4. Pretakati odgovor agenta po posameznih žetonih.


## Nastavitev

Pred zagonom tega zvezka se prepričajte, da imate:

1. **Projekt Azure AI Foundry** z nameščenim modelom klepeta (npr. `gpt-4o-mini`).
2. **Prijavljeni v Azure CLI** — zaženite `az login` v terminalu.
3. **Nastavljene zahtevane spremenljivke okolja:**
   - `AZURE_AI_PROJECT_ENDPOINT` — vaš konec projekta Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ime vašega nameščenega modela.

Spodnja celica namesti potrebne Python pakete.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Ustvarjanje vašega prvega agenta

Agent potrebuje dve stvari:

- **Navodila**, ki mu povejo *kdo je* in *kako naj se obnaša* (sistemski poziv).
- **Orodja** — funkcije v Pythonu, okrašene z `@tool`, ki jih agent lahko pokliče za pridobivanje informacij ali izvajanje dejanj.

Spodaj definiramo preprosto orodje, ki vrne seznam priljubljenih počitniških destinacij. Agent bo to orodje uporabil, ko uporabnik zaprosi za priporočila glede potovanj.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Za bolj interaktivno izkušnjo lahko **streamate** odziv agenta. Namesto da bi čakali na celoten odgovor, agent sproti posreduje koščke besedila, ko jih ustvari. To je posebej uporabno v klepetalnih vmesnikih, kjer želite prikazati izhod v realnem času.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Povzetek

V tej lekciji ste se naučili, kako:

- **Ustvariti ponudnika**, ki se poveže z Azure AI Foundry Agent Service preko `AzureAIProjectAgentProvider`.
- **Določiti orodje** z uporabo dekoratorja `@tool`, da lahko agent kliče vaše Python funkcije.
- **Zagnati agenta** z uporabniškim sporočilom in izpisati njegov odgovor.
- **Pretakati odgovore** za izhod v realnem času.

V naslednji lekciji bomo poglobljeno raziskali agentne okvire in se naučili, kako agentom zagotoviti močnejša orodja in večstopenjske sposobnosti sklepanja.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
